# EvalPlus benchmark: Qwen2.5-Coder-7B (base) vs v3 (merged)

HumanEval + HumanEval+ (EvalPlus), vLLM, greedy pass@1.

**Trước khi chạy (Kaggle):**
1. Settings → Accelerator: **GPU T4 x2**. Settings → Internet: **On**.
2. Add-ons → Secrets: thêm `HF_TOKEN` (quyền đọc, repo v3-merged đang private) và attach vào notebook.

Model 7B fp16 (~15GB) không vừa 1 T4 nên dùng `--tp 2`; T4 không có bf16 nên dùng `float16`.


## 1. Cài đặt

In [ ]:
!pip install -q uv
!uv venv /kaggle/working/vllm_env --python 3.12
!uv pip install -q --python /kaggle/working/vllm_env/bin/python "evalplus[vllm]"
!/kaggle/working/vllm_env/bin/python -c "import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| vllm', vllm.__version__)"

## 2. Config

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

BASE = "Qwen/Qwen2.5-Coder-7B"
V3   = "hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged"
TP, DTYPE = 2, "float16"
EVAL = "/kaggle/working/vllm_env/bin/evalplus.evaluate"

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Base (completion prompt)

In [ ]:
!{EVAL} --model {BASE} --dataset humaneval --backend vllm --greedy --force-base-prompt --tp {TP} --dtype {DTYPE} 2>&1 | tee evalplus_base.log | tail -n 40

## 4. v3 (chat template)

In [ ]:
!{EVAL} --model {V3} --dataset humaneval --backend vllm --greedy --tp {TP} --dtype {DTYPE} 2>&1 | tee evalplus_v3.log | tail -n 40

## 5. Summary

In [ ]:
!grep -E "pass@1|humaneval" evalplus_base.log evalplus_v3.log

## Nếu lỗi

- `CUDA out of memory`: thêm `--gpu-memory-utilization 0.85` hoặc giảm `--max-model-len`, hoặc chạy lại sau khi Restart Session.
- vLLM không khởi động được trên T4 (lỗi attention backend): thử `os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"` trong cell Config.
- Cell cài đặt lỗi (uv không tải được Python 3.12): đổi `--python 3.12` thành `--python 3.13` hoặc bỏ tham số.
- Điểm v3 thấp bất thường: chat template của Qwen tự thêm system prompt mặc định, trong khi lúc train không có.
